# Actividad individual 2 — Descubriendo patrones de tráfico con datos reales
## TC2035 — Diseño de Redes Neuronales y Aprendizaje Profundo

**Nombre:** Emilio Alonso Sánchez  
**Matrícula:** A00841853  
**Grupo:** *Completar*  
**Modalidad:** Individual  
**Herramienta:** Google Colab (Python)  
**Reto:** Predicción inteligente de tiempos de verde — SEMEX

### Introducción
Analizaré datos reales de la intersección 01 del Distrito Tec en Monterrey para identificar patrones antes de construir una red neuronal. El archivo **`Int. 01 mañana.xlsx`** contiene 16 hojas: **`Hora Máx.`**, 14 movimientos (numerados del 1 al 14) y **`Vol. Dir.`**. Para esta actividad se compararán los movimientos **1, 2 y 3**, registrados en intervalos de **15 minutos** durante 07:00–10:00. Se realizarán visualizaciones, normalización Min-Max, interpretaciones y una reflexión para el reto SEMEX. **No se entrena una MLP** en esta etapa.

### Contexto de la intersección según el croquis

El croquis del estudio reporta los siguientes volúmenes durante la **hora de máxima demanda de cada periodo**. No deben confundirse con todos los vehículos observados durante las tres horas de cada levantamiento.

| Periodo | Levantamiento | Hora de máxima demanda | Volumen total |
|---|---|---|---:|
| Mañana | 07:00–10:00 | 08:00–09:00 | 1,771 vehículos |
| Tarde | 12:00–15:00 | 13:45–14:45 | 2,287 vehículos |
| Noche | 17:00–20:00 | 17:15–18:15 | 2,372 vehículos |

En la lámina de la **mañana** se muestran cuatro valores de acceso que suman 1,771: 454, 305, 548 y 464 vehículos. También se representan movimientos alrededor de la rotonda en la zona de Av. Eugenio Garza Sada, Av. Fernando García Roel, Del Estado y el Tecnológico de Monterrey.

**Precaución:** el croquis permite contextualizar los accesos, pero la correspondencia de una columna particular del Excel con una trayectoria del croquis debe verificarse con los encabezados y la codificación original. No se asignan direcciones o giros inventados.

---

## Parte 1. Exploración y comprensión de los datos

### 1.1 Preparación y carga del archivo

Este notebook **intenta descargar** el Excel desde la rama \`main\` de \`A00841853/Redes-Neuronales\`. Si todavía no está disponible en GitHub, permite seleccionarlo con el botón de carga de Colab. En VS Code/Jupyter local también puede leerlo si está en la misma carpeta del notebook.

Si los registros se organizaron con encabezados combinados o varias tablas en una hoja, las siguientes celdas mostrarán las filas sin procesar antes de elegir los movimientos. **No se rellenan con datos simulados**.

In [ ]:
# Bibliotecas: Colab incluye habitualmente estas dependencias
import re
from pathlib import Path
from datetime import datetime, time
from io import BytesIO
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 180)
np.set_printoptions(suppress=True, precision=4)

ARCHIVO = "Int. 01 mañana.xlsx"
URL = "https://raw.githubusercontent.com/A00841853/Redes-Neuronales/main/" + quote(ARCHIVO)

archivo_excel = None
if Path(ARCHIVO).is_file():
    archivo_excel = Path(ARCHIVO)
    print("Excel encontrado localmente:", archivo_excel)
else:
    import requests
    try:
        respuesta = requests.get(URL, timeout=15)
        if respuesta.ok and respuesta.content[:2] == b"PK":
            archivo_excel = BytesIO(respuesta.content)
            print("Excel descargado de GitHub:", URL)
        else:
            print("El archivo Excel aún no está accesible en GitHub (código:", respuesta.status_code, ").")
    except requests.RequestException as exc:
        print("No se pudo acceder a GitHub:", exc)

if archivo_excel is None:
    try:
        from google.colab import files
        print("Selecciona ahora el archivo:", ARCHIVO)
        subidos = files.upload()
        rutas = [nombre for nombre in subidos if nombre.lower().endswith((".xlsx", ".xls"))]
        if not rutas:
            raise FileNotFoundError("No seleccionaste un archivo Excel.")
        archivo_excel = BytesIO(subidos[rutas[0]])
        print("Archivo cargado:", rutas[0])
    except ImportError as exc:
        raise FileNotFoundError(
            "No se encontró el Excel. Pon 'Int. 01 mañana.xlsx' en la carpeta del notebook, "
            "o ejecuta este notebook en Google Colab para subir el archivo."
        ) from exc

libro = pd.ExcelFile(archivo_excel)
print("\nHojas disponibles:", libro.sheet_names)
hojas_crudas = {
    nombre: pd.read_excel(libro, sheet_name=nombre, header=None, dtype=object)
    for nombre in libro.sheet_names
}
for nombre, tabla in hojas_crudas.items():
    print(f"Hoja: {nombre!r} | filas={tabla.shape[0]}, columnas={tabla.shape[1]}")

In [ ]:
# La estructura REAL del Excel: 16 hojas, 14 de movimientos numerados
for nombre in libro.sheet_names:
    tabla = hojas_crudas[nombre]
    print(f"Hoja {nombre!r}: {tabla.shape[0]} filas, {tabla.shape[1]} columnas")

print("\nVista inicial de la hoja de movimiento 1:")
display(hojas_crudas["1"].iloc[:11, :17].fillna(""))
print("\nEn las hojas 1, 2 y 3: A=hora inicial, B=hora final, Q=TOTAL.")
print("Las filas 14 y 19 separan bloques de registro; no son datos de aforo.")

### Interpretación 1. Comprendiendo los datos

**¿Qué representa cada registro?** Una fila válida corresponde a un **intervalo de 15 minutos** y al total de vehículos observados en un movimiento; no representa un vehículo individual.

**¿Qué variables identifiqué?** Los tiempos de **inicio** y **fin** del intervalo (columnas A y B), categorías vehiculares intermedias y el **TOTAL** del movimiento (columna Q). Cada movimiento aparece en una hoja identificada con su número, del 1 al 14.

**¿Qué unidades tienen los volúmenes?** **Vehículos por intervalo de 15 minutos**. Sumando cuatro registros consecutivos se obtiene el volumen de una hora.

**¿Qué puede ser útil?** El comportamiento por intervalo, los momentos de máxima demanda y las diferencias entre movimientos. También serían relevantes ocupación y colas si se recopilan en etapas posteriores.

**¿Qué limpieza se requirió?** El archivo tiene encabezados especiales y filas separadoras **14 y 19**, que no son conteos. La lectura se realiza con `header=None` y se conservan solo filas con horas de inicio y fin identificables. En los movimientos 1, 2 y 3 se obtienen **12 intervalos completos, sin valores faltantes en Q**. No se imputan datos inexistentes.

### 1.2. Identificación de intervalos y movimientos

La inspección del archivo confirma que los movimientos 1, 2 y 3 se encuentran en **hojas distintas**, no en tres columnas de una misma hoja. Las horas están en **A y B** y los conteos en **Q**, bajo el encabezado `TOTAL`. Para evitar confundir categorías de vehículo con movimientos, las siguientes celdas extraen solo las columnas pertinentes de cada hoja y las unen por intervalo.

Se analizarán 12 intervalos: 07:00–07:15, 07:15–07:30, ..., 09:45–10:00.

In [ ]:
# Lectura de horas Excel: pueden llegar como datetime.time o fracciones de día
def leer_hora(valor):
    if pd.isna(valor):
        return None
    if isinstance(valor, (datetime, time, pd.Timestamp)):
        return f"{valor.hour:02d}:{valor.minute:02d}"
    if isinstance(valor, (int, float, np.number)) and not isinstance(valor, bool):
        v = float(valor)
        if 0 <= v < 1:
            minutos = round(v * 1440) % 1440
            return f"{minutos//60:02d}:{minutos%60:02d}"
    if isinstance(valor, str):
        import re
        coincidencia = re.match(r"^\s*(\d{1,2}):(\d{2})", valor)
        if coincidencia:
            return f"{int(coincidencia.group(1)):02d}:{coincidencia.group(2)}"
    return None

def leer_movimiento(numero):
    # Tres hojas separadas; en cada una se leen A, B y Q
    crudo = hojas_crudas[str(numero)]
    tabla = crudo.iloc[:, [0, 1, 16]].copy()
    nombre = f"Movimiento {numero}"
    tabla.columns = ["Intervalo", "Fin", nombre]
    tabla["Intervalo"] = tabla["Intervalo"].map(leer_hora)
    tabla["Fin"] = tabla["Fin"].map(leer_hora)
    tabla[nombre] = pd.to_numeric(tabla[nombre], errors="coerce")
    tabla = tabla.loc[tabla["Intervalo"].notna() & tabla["Fin"].notna()].copy()
    tabla = tabla.loc[tabla["Intervalo"].str[:2].astype(int).between(7, 9)]
    if tabla[nombre].isna().any():
        raise ValueError(f"Hay valores de volumen faltantes en {nombre}")
    return tabla.reset_index(drop=True)

print("Lectura preparada para las hojas 1, 2 y 3.")

---

## Parte 2. Selección y visualización de movimientos vehiculares

Se seleccionaron los **movimientos 1, 2 y 3**, tal como están identificados en las hojas del archivo. Se utilizará su columna **Q (TOTAL)** y los mismos 12 intervalos de 15 minutos. En la hora pico matutina (08:00–09:00) estos movimientos registran, respectivamente, **291, 179 y 78 vehículos**, que suman **548** y corresponden al acceso de 548 vehículos señalado en el croquis.

No se atribuyen giros o calles específicas a cada número de movimiento sin la clave original de equivalencias del estudio.

In [ ]:
# Construir la tabla comparativa utilizando las tres hojas independientes
movimientos = ["Movimiento 1", "Movimiento 2", "Movimiento 3"]
series = [leer_movimiento(n) for n in (1, 2, 3)]
datos = series[0]
for serie in series[1:]:
    datos = datos.merge(serie, on=["Intervalo", "Fin"], validate="one_to_one")

datos["Orden"] = datos["Intervalo"].str[:2].astype(int)*60 + datos["Intervalo"].str[3:5].astype(int)
datos = datos.sort_values("Orden").drop(columns="Orden").reset_index(drop=True)
faltantes = datos[movimientos].isna().sum()

assert len(datos) == 12
assert not faltantes.any()
assert datos["Intervalo"].iloc[0] == "07:00"
assert datos["Intervalo"].iloc[-1] == "09:45"
display(Markdown("### Registros reales por intervalo"))
display(datos)
print("Registros válidos:", len(datos))
print("Conteos faltantes:", faltantes.to_dict())

In [ ]:
# Sumar cuatro intervalos para obtener el flujo de 08:00–09:00
minuto = datos["Intervalo"].str[:2].astype(int)*60 + datos["Intervalo"].str[3:5].astype(int)
hora_pico = datos.loc[minuto.between(480, 539), movimientos].sum()
total_3h = datos[movimientos].sum()
print("Volúmenes de 07:00 a 10:00:", total_3h.to_dict())
print("Volúmenes durante 08:00 a 09:00:", hora_pico.to_dict())
print("Suma de los tres durante hora pico:", int(hora_pico.sum()))
assert total_3h.astype(int).tolist() == [842, 491, 188]
assert hora_pico.astype(int).tolist() == [291, 179, 78]
assert int(hora_pico.sum()) == 548

### 2.2. Correspondencia con el croquis

Los movimientos 1, 2 y 3 presentan **291, 179 y 78 vehículos** en la hora de máxima demanda, de **08:00 a 09:00**. En conjunto representan el acceso de **548 vehículos** dibujado en la parte inferior del croquis de la rotonda. Se trata de tres corrientes del mismo acceso, registradas por separado en el Excel.

El croquis muestra **1,771 vehículos en toda la intersección** durante esa hora. Por ello, los 548 vehículos seleccionados no equivalen a todo el tráfico del cruce, y los totales de las tres horas tampoco se deben confundir con el volumen de una sola hora.

In [ ]:
# Gráfica 1: evolución por intervalo de los tres movimientos
fig, ax = plt.subplots(figsize=(11, 5))
for movimiento in movimientos:
    ax.plot(datos["Intervalo"], datos[movimiento],
            marker="o", linewidth=2, markersize=5, label=movimiento)
ax.set_title("Intersección 01 | Volumen por intervalo — tres movimientos (mañana)")
ax.set_xlabel("Inicio del intervalo de medición")
ax.set_ylabel("Vehículos por intervalo registrado")
ax.grid(alpha=0.25)
ax.legend(title="Serie del Excel")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica 2: volúmenes totales observados en los intervalos del Excel
totales = datos[movimientos].sum(axis=0, skipna=True)
fig, ax = plt.subplots(figsize=(8.5, 4.5))
barras = ax.bar(movimientos, totales.values, color=["#2563eb", "#059669", "#d97706"])
for b, v in zip(barras, totales.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{v:,.0f}",
            ha="center", va="bottom", fontsize=10)
ax.set_ylim(0, max(totales.values) * 1.17)
ax.set_ylabel("Vehículos registrados (suma de intervalos disponibles)")
ax.set_title("Comparación de los tres movimientos seleccionados")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# Comparación de los movimientos: máximo por serie y máximo conjunto por intervalo
datos["Suma de los tres"] = datos[movimientos].sum(axis=1, min_count=1)
mov_mayor = totales.idxmax()
intervalo_max = datos.loc[datos["Suma de los tres"].idxmax(), "Intervalo"]
print("Mayor volumen acumulado:", mov_mayor, f"({totales[mov_mayor]:,.0f} vehículos)")
print("Intervalo con mayor suma entre las tres series:", intervalo_max)
print("NOTA: estos resultados corresponden SOLO a las tres columnas seleccionadas.")
print("No deben confundirse con la hora de máxima demanda de toda la intersección.")

### Interpretación 2. Comportamiento del tráfico

- **Movimiento con mayor volumen:** se identifica mediante la suma de sus registros en la tabla y la segunda gráfica. El resultado numérico aparece inmediatamente arriba.
- **Intervalo de mayor demanda:** se calcula como el intervalo con la mayor **suma de los tres movimientos seleccionados**, no como la hora de máxima demanda de toda la intersección. Es necesario diferenciar ambos conceptos.
- **¿Se comportan igual?** No debe suponerse que sí: la gráfica de líneas permite contrastar picos, disminuciones y variaciones entre series. Sus diferencias se cuantifican en la celda siguiente.
- **¿Por qué importan esas diferencias?** Si todos los movimientos recibieran tiempos de verde iguales, podrían producirse esperas excesivas en unos y tiempo desaprovechado en otros. Sin embargo, una mayor demanda por sí sola no determina un split: también importan la saturación, los conflictos, las colas y los peatones.

**Referencia adicional:** el croquis informa que la hora de máxima demanda matutina del conjunto de la intersección fue 08:00–09:00, con 1,771 vehículos; este valor no necesariamente coincide con el máximo puntual de nuestras tres series.

In [ ]:
# Interpretación numérica directamente vinculada con los datos procesados
resumen = pd.DataFrame({
    "Volumen total": totales,
    "Máximo por intervalo": datos[movimientos].max(),
    "Intervalo del máximo": [
        datos.loc[datos[m].idxmax(), "Intervalo"] for m in movimientos
    ],
    "Desviación estándar": datos[movimientos].std()
})
display(resumen.round(2))
orden = totales.sort_values(ascending=False)
print(f"El movimiento de mayor volumen es {orden.index[0]} ({orden.iloc[0]:,.0f} vehículos).")
print(f"El de menor volumen es {orden.index[-1]} ({orden.iloc[-1]:,.0f} vehículos).")
print(f"La mayor suma de las tres series se produce en el intervalo {intervalo_max}.")
print("La desviación estándar resume la variabilidad observada de cada movimiento.")

---

## Parte 3. Normalización de los datos

### Transformación Min-Max

Se usa la función, calculada **por separado para cada movimiento**, sobre los intervalos efectivamente disponibles:

\[
x_{\text{normalizado}}=\frac{x-x_{\min}}{x_{\max}-x_{\min}}.
\]

Los valores originales se conservan para mostrar las unidades de tráfico. La escala transformada sirve únicamente para **comparar formas y cambios relativos** en una misma gráfica.

**Volumen constante:** si \(x_{\max}=x_{\min}\), la expresión tiene denominador cero. Para fines de esta actividad, todos los registros válidos de esa columna se asignan a 0 y se deja constancia de que la serie no presenta variación. No se convierten en cero los registros faltantes.

In [ ]:
# Normalización Min-Max por serie, con manejo explícito de constantes y NaN
normalizados = pd.DataFrame(index=datos.index)
constantes = []
for movimiento in movimientos:
    serie = datos[movimiento].astype(float)
    minimo, maximo = serie.min(), serie.max()
    if pd.isna(minimo):
        raise ValueError(f"El {movimiento} no tiene valores válidos.")
    if np.isclose(maximo, minimo):
        normalizados[movimiento] = serie.apply(lambda x: np.nan if pd.isna(x) else 0.0)
        constantes.append(movimiento)
    else:
        normalizados[movimiento] = (serie - minimo) / (maximo - minimo)

tabla_normalizacion = pd.DataFrame({"Intervalo": datos["Intervalo"]})
for movimiento in movimientos:
    tabla_normalizacion[f"{movimiento} original"] = datos[movimiento]
    tabla_normalizacion[f"{movimiento} Min-Max"] = normalizados[movimiento]

display(Markdown("### Tabla de valores originales y normalizados"))
display(tabla_normalizacion.round(4))
print("Movimientos constantes:", constantes if constantes else "ninguno")

valores_normalizados = normalizados.to_numpy(dtype=float)
assert np.all((valores_normalizados[np.isfinite(valores_normalizados)] >= -1e-10))
assert np.all((valores_normalizados[np.isfinite(valores_normalizados)] <= 1 + 1e-10))

In [ ]:
# Gráfica 3: comparar las tres series normalizadas
fig, ax = plt.subplots(figsize=(11, 5))
for movimiento in movimientos:
    ax.plot(datos["Intervalo"], normalizados[movimiento],
            marker="o", linewidth=2, markersize=5, label=movimiento)
ax.set_title("Intersección 01 | Volúmenes normalizados con Min-Max")
ax.set_xlabel("Inicio del intervalo de medición")
ax.set_ylabel("Volumen normalizado (0 a 1)")
ax.set_ylim(-0.06, 1.07)
ax.grid(alpha=0.25)
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("La primera gráfica muestra vehículos por intervalo; esta tercera gráfica muestra escala relativa.")

### Interpretación 3. Normalización

- **¿Qué cambió?** El volumen dejó de expresarse en vehículos y se transformó a una escala numérica de 0 a 1 **para cada movimiento por separado**. El mínimo observado se representa con 0 y el máximo con 1, salvo en series constantes.
- **¿Se conservan los patrones?** Sí: cuando el rango no es cero, Min-Max es una transformación lineal creciente, así que conserva el orden temporal, la ubicación de máximos/mínimos y la forma de crecimiento o disminución, aunque cambia la escala vertical.
- **¿Por qué sirve antes de una MLP?** Permite que magnitudes distintas tengan rangos comparables. Puede mejorar la estabilidad y eficiencia del ajuste de los pesos y evitar que una entrada numéricamente grande domine solo por su escala.
- **¿Todos tienen el mismo volumen?** No. Un 1 en dos series diferentes puede representar volúmenes absolutos distintos. Es necesario consultar los datos originales para comparar cantidades de vehículos.
- **¿Qué pasa si hay un movimiento constante?** Su valor normalizado se fija en 0 y se documenta. Una serie constante no aporta variación en estos registros.

**Prevención de fuga de información:** esta normalización usa los registros para una exploración didáctica. En una MLP real, mínimos y máximos se calculan únicamente con el conjunto de entrenamiento, y después se aplican sin reajuste a validación, prueba y operación.

---

## Parte 4. Identificación de patrones para una futura MLP

### Propuesta de variables de entrada

| Variable candidata | Información que aportaría |
|---|---|
| Volumen de movimiento 1, vehículos por intervalo | Cuánta demanda llega a esa trayectoria durante un periodo definido. |
| Volumen de movimiento 2, vehículos por intervalo | Demanda concurrente que puede competir por tiempo de servicio. |
| Volumen de movimiento 3, vehículos por intervalo | Diferencias de carga entre movimientos y posibles cambios de prioridad. |
| Hora del día codificada numéricamente | Patrones horarios de llegada (preferiblemente codificación cíclica para todo el día). |
| Ocupación de carril y longitud de cola (datos adicionales) | Estado de congestión que los conteos de paso, por sí solos, no muestran. |

### Relaciones que podría aprender

La red podría identificar franjas con más circulación, variaciones de demanda entre trayectorias y relaciones entre volumen, ocupación y acumulación de colas. Para que prediga **tiempos de verde NS/EO o splits por fase**, harían falta pares consistentes de **entradas** (estado del tráfico) y **objetivos** (tiempos de verde de referencia válidos o determinados mediante una política optimizada).

No sería metodológicamente correcto fabricar etiquetas de tiempos de verde a partir de estos conteos sin un criterio de control y sin comprobar la seguridad y las restricciones de la intersección.

### Interpretación 4. Conexión con las redes neuronales

- **¿Qué datos del estudio usaría como entradas?** Los tres volúmenes por intervalo seleccionados, sus marcas de tiempo y, si existen en otras columnas debidamente documentadas, más movimientos del mismo instante. También buscaría incorporar ocupación, longitud de cola y flujos peatonales.
- **¿Qué me gustaría predecir?** Duraciones de verde por fase o dirección que sean seguras, operativamente factibles y útiles para disminuir las demoras.
- **¿Contamos con las etiquetas objetivo?** No se han identificado en estos datos de conteo etiquetas verificadas de tiempos de verde recomendados. Para entrenar un modelo supervisado de regresión se necesitarían esas etiquetas y el método usado para obtenerlas.
- **¿Observar un patrón equivale a que la red lo aprendió?** No. Una gráfica permite describir datos ya observados. Para afirmar que un modelo aprendió una relación, debería entrenarse con ejemplos adecuados y mantener capacidad predictiva en otros intervalos o días no utilizados en el ajuste.
- **¿Cómo comprobaría que generaliza?** Separaría los datos por tiempo o por día, reservaría un conjunto de prueba no visto, ajustaría el escalador solo con entrenamiento y reportaría MAE/RMSE de los splits. Después compararía las estrategias de control en una simulación calibrada y analizaría demora, colas, detenciones y seguridad.

La MLP no se implementa ni entrena en esta actividad, conforme a las instrucciones.

### Verificación final de la exploración
Las comprobaciones permiten confirmar que el análisis se realizó sobre tres columnas de datos cargados del Excel. Un resultado correcto **no garantiza por sí solo** que la correspondencia espacial con el croquis haya sido validada: esa revisión depende del significado de los encabezados originales.

In [ ]:
# Resumen técnico reproducible del análisis
assert len(movimientos) == 3
assert len(columnas_seleccionadas) == 3
assert datos["Intervalo"].notna().all()
assert set(movimientos).issubset(datos.columns)
assert set(movimientos).issubset(normalizados.columns)

print("Archivo fuente:", ARCHIVO)
print("Hoja de procedencia:", hoja_analisis)
print("Registros analizados:", len(datos))
print("Columnas del Excel:", [columna_excel(c) for c in columnas_seleccionadas])
print("Valores faltantes:", faltantes.to_dict())
print("Mayor suma de los tres movimientos por intervalo:", intervalo_max)
print("Mayor total de un movimiento:", mov_mayor)
print("Estado: análisis descriptivo completado; ninguna MLP ha sido entrenada.")

---

## Conclusión: ¿qué aprendí sobre los datos de tráfico?

Esta actividad me permitió comprender que, antes de diseñar una red neuronal para el control semafórico, es necesario examinar cómo están organizados los datos y qué representa cada medición. El croquis de la intersección 01 muestra que la demanda no permanece constante a lo largo del día: durante las horas de máxima demanda se registran 1,771 vehículos por la mañana, 2,287 por la tarde y 2,372 por la noche. Sin embargo, para estudiar movimientos individuales es indispensable revisar los intervalos del archivo Excel y no utilizar esos totales generales como si fueran conteos de cada giro.

Al comparar tres movimientos, las tablas y gráficas permiten identificar diferencias de volumen y momentos de mayor actividad. La normalización Min-Max facilita comparar sus variaciones relativas en una escala común, pero no significa que transporten la misma cantidad de vehículos.

También comprendí que encontrar patrones en los datos no equivale a construir un modelo confiable. Para el reto de SEMEX será necesario disponer de más observaciones, conocer las restricciones del semáforo y obtener tiempos de verde objetivo debidamente justificados. Solo después tendría sentido entrenar una MLP y evaluar si sus recomendaciones realmente reducen las demoras.